# 5 Laufzeitmodell

# 5.2 Kompilieren und Ausführen

## Python als Mischmodell (8)

```text
 Quelltext
   |
   v
 AST (Abstract Syntax Tree)
   |
   v
 Bytecode
   |
   v
 Python Virtual Machine
```

In [1]:
import ast

source = "x = 1 + 2 * 3"
tree = ast.parse(source)
print(ast.dump(tree, indent=2))

Module(
  body=[
    Assign(
      targets=[
        Name(id='x', ctx=Store())],
      value=BinOp(
        left=Constant(value=1),
        op=Add(),
        right=BinOp(
          left=Constant(value=2),
          op=Mult(),
          right=Constant(value=3))))])


In [2]:
import dis

code = compile(source, "<demo>", "exec")
dis.dis(code)

  0           RESUME                   0

  1           LOAD_SMALL_INT           7
              STORE_NAME               0 (x)
              LOAD_CONST               1 (None)
              RETURN_VALUE


## Laufzeitobjekte in Python (9)

In [3]:
def greet(name):
    return f"Hallo {name}"

alias = greet
greet.category = "demo"

print(alias("Ada"))
print("function object attribute:", greet.category)

Hallo Ada
function object attribute: demo


In [4]:
class Counter:
    pass

Counter.version = "1.0"

print("class attribute:", Counter.version)

class attribute: 1.0


In [5]:
car_counter = Counter()
car_counter.color = "red"

print("instance attribute:", car_counter.color)

instance attribute: red


In [6]:
bike_counter = Counter()
bike_counter.version = "2.0"

print("class attribute:", Counter.version)
print("instance attribute:", bike_counter.version)

class attribute: 1.0
instance attribute: 2.0


In [7]:
def make_greet(greeting):
    def greet(name):
        return f"{greeting}, {name}"
    return greet

greet_english = make_greet("Hello")
greet_german = make_greet("Guten Tag")

print(greet_english("Alice"))
print(greet_german("Sophie"))

Hello, Alice
Guten Tag, Sophie


# 5.3 Speicherbereiche

## Call Stack (11)

In [8]:
import inspect

def print_stack():
    for i, frame in enumerate(inspect.stack()):
        print(f"{i}: {frame.function}() in {frame.filename}:{frame.lineno}, {frame.frame.f_locals}")
        if i >= 5:
            print("...")
            break

def factorial(n):
    if n == 1:
        # raise Exception("This is an error")
        # print_stack()
        return 1
    return n * factorial(n - 1)

factorial(10)

3628800

## Stack und Heap in Python (13)

Beispiel:

```text
      Stack                           Heap

Frame build_numbers():
  numbers (Name) -----------------> [1, 2, 3] (Objekt)
Frame ipykernel():                |
  result (Name) -------------------
```


In [9]:
def build_numbers():
    numbers = [1, 2, 3]
    print("inside function:", id(numbers), numbers)
    return numbers

result = build_numbers()

print("outside function:", id(result), result)

inside function: 2016620784960 [1, 2, 3]
outside function: 2016620784960 [1, 2, 3]


# 5.4 Objekterzeugung und Lebensdauer

## Lebensdauer von Objekten (17)

In [10]:
import gc
import weakref

class Box:
    def __init__(self, value):
        self.value = value

box = Box([1, 2, 3])
alias = box
ref = weakref.ref(box)

del box
print("after del box, object alive:", ref() is not None)

del alias
gc.collect()
print("after del alias, object alive:", ref() is not None)

after del box, object alive: True
after del alias, object alive: False


# 5.5 Garbage Collection

## Zyklische Garbage Collection (21)

In [11]:
import gc
import weakref

class Node:
    def __init__(self, name):
        self.name = name
        self.next = None

gc.disable()

a = Node("A")
b = Node("B")
a.next = b
b.next = a

wa = weakref.ref(a)
wb = weakref.ref(b)

del a
del b
print("before gc.collect():", wa() is not None, wb() is not None)

gc.collect()
print("after gc.collect():", wa() is not None, wb() is not None)

gc.enable()

before gc.collect(): True True
after gc.collect(): False False


## Externe Ressourcen (22)

In [12]:
from pathlib import Path

tmp_path = Path("runtime_model_demo.txt")
with open(tmp_path, "w", encoding="utf-8") as handle:
    handle.write("temporary data\n")
    print("inside with, handle.closed =", handle.closed)

print("outside with, handle.closed =", handle.closed)
tmp_path.unlink()

inside with, handle.closed = False
outside with, handle.closed = True


# 5.6 Explizite Speicherverwaltung

## Destruktoren und RAII (25)

Das RAII-Muster (Resource Acquisition Is Initialization) in C++ garantiert automatische Speicherfreigabe durch Destruktoren:

```c++
class Resource {
    private:
        int* data;

    public:
        // Konstruktor: Ressourcen erwerben
        Resource(int size) {
            data = new int[size];
        }

        // Destruktor: Ressourcen automatisch freigeben
        ~Resource() {
            delete[] data;
        }
};

int main() {
    {
        Resource res(100);
        // res wird hier verwendet
    }
    // Destruktor wird AUTOMATISCH aufgerufen, wenn Block endet.
    // Speicher ist jetzt freigegeben.

    return 0;
}
```

## Typische Fehler (26)

Typische Fehler ohne automatische Speicherverwaltung:

```c++
int main() {
    // Fehlerfall 1: Memory Leak
    int* ptr1 = new int[1000];
    // Speicher wird allokiert, aber NICHT freigegeben.
    // Wenn ptr1 aus dem Scope geht: Speicher bleibt belegt

    // Fehlerfall 2: Dangling Pointer
    int* ptr2 = new int[1000];
    delete ptr2;
    std::cout << *ptr2;
    // FEHLER: Pointer zeigt auf ungültigen Speicher.
    // Undefined behavior, evtl Segmentation Fault

    // Fehlerfall 3: Double Delete
    int* ptr3 = new int[1000];
    delete ptr3;
    delete ptr3;
    // FEHLER: Speicher wurde schon freigegeben.
    // Undefined behavior, evtl Crash

    return 0;
}
```